In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
  OneHotEncoder,
  StandardScaler,
  KBinsDiscretizer
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Preparing the Housing Prices Dataset

## Dataset Description

Columns and their data types:
- longitude: continuous numerical
- latitude: continuous numerical
- housing_median_age: integer, needs to be categorized due to a value cap in the data collection
- total_rooms: integer
- total_bedrooms: integer
- population: integer
- households: integer
- median_income: continuous, needs to be categorized due to value cap in the data collection
- ocean_proximity: non-linear categorical
- median_house_value: continuous, needs to be categorized due to value cap in the data collection\*

\* Since the median_house_value is the target for the model's predictions, this one will not be categorized, we'll simply accept the model's incapacity to estimate accurately the value of highly valued housing

In [3]:
df = pd.read_csv("housing.csv")
df.dropna(inplace=True)
df_x = df.drop(columns=["median_house_value"])
df_y = df[["median_house_value"]]

numeric_columns = [
  "longitude",
  "latitude",
  "total_rooms",
  "total_bedrooms",
  "population",
  "households",
]

categorical_columns = [
  "ocean_proximity"
]

capped_columns = [
  "housing_median_age",
  "median_income",
]

In [ ]:
# Split training and testing dataset
train_x, test_x, train_y, test_y = train_test_split(df_x, df_y, test_size=0.1)

# Preprocessing
preprocessor = ColumnTransformer(
  transformers=[
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(drop="first"), categorical_columns),
    ("bin", KBinsDiscretizer(n_bins=10, encode="onehot-dense", strategy="quantile", quantile_method="linear"), capped_columns)
  ]
)

output_preprocessor = ColumnTransformer(
  transformers=[
    ("num", StandardScaler(), ["median_house_value"]) 
  ]
)

preprocessor.fit(train_x)
train_x = preprocessor.transform(train_x)
test_x = preprocessor.transform(test_x)

output_preprocessor.fit(train_y)
train_y = output_preprocessor.transform(train_y)
test_y = output_preprocessor.transform(test_y)

# Linear Regression

In [65]:
class LinearRegression:
  def __init__(
    self,
    input_size: int
  ):
    self.weights: np.ndarray = np.random.rand(input_size)
    self.bias: float = np.random.rand(1)[0]

  def loss(self, output: np.ndarray, expected: np.ndarray) -> float:
    return np.sum((output-expected)**2)/2

  def fit(self, train_x, train_y, verbose=False, epochs: int=100, training_step: float=0.01, plot: bool=False):
    losses = []
    for epoch in range(epochs):
      loss = 0
      step = np.zeros(self.weights.shape[0])
      bias_step = 0
      for input, expected in zip(train_x, train_y):
        output = self.predict(input)
        loss += self.loss(output, expected)
        step += training_step*(output-expected)*input
        bias_step += training_step*(output-expected)
      loss /= len(train_x)
      self.weights -= step/len(train_x)
      self.bias -= bias_step/len(train_x)
      losses.append(loss)
      if verbose: print(f"Epoch {epoch}: loss: {loss}")
    if plot:
      plt.plot(losses)

  def predict(self, input: np.ndarray):
    return self.weights @ input + self.bias

  def test(self, test_x, test_y):
    loss = 0
    for input, expected in zip(test_x, test_y):
      loss += self.loss(self.predict(input), expected)
    loss /= len(test_x)
    print(f"Tested loss: {loss}")

## Testing the implementation

In [ ]:
# Training model on
model = LinearRegression(train_x[0].shape[0])
model.fit(train_x, train_y, verbose=False, epochs=400, training_step=0.01, plot=True)
model.test(test_x, test_y)